# Week 7 v2 — Notebook 2A: ADCD-Net on every image (shrink512 and full)

**What it does:** reads the dataset `PATH_TO_W7_COMMON_SETS_DATASET` (the page sets from notebook 1) and runs the pre-trained ADCD-Net on
**every image** of the manifest at two sizes: `shrink512` (long side 512 px) and `full` (full resolution, cut in 512 × 512
pieces and stitched back). It saves one **result map per image and size** in the standard format that notebook 3 reads.

**Status:** the SMOKE test passed on Kaggle (62 images, 9 checks, 0 missing keys). **This copy is set to `SMOKE = False`: use Save & Run All.**

**How to run (Kaggle):** attach the datasets `PATH_TO_W7_COMMON_SETS_DATASET`, `PATH_TO_ADCD_WEIGHTS_DATASET` and `PATH_TO_ADCD_RUN_OUTPUTS_DATASET` (it holds
`qt_table.pk` and the Week 5 OCR masks); Accelerator = **GPU T4 x2**; Internet **ON**. First run with `SMOKE = True`
(3 pages per set, about 10 minutes, mostly installing), read the printed estimate, then set `SMOKE = False` in the first
code cell and use **Save & Run All**. Nothing is typed in; every input is found automatically.

**What you get:** zip 3 (`w7_2A_3_NEXT_ROUND_adcd-results.zip`) = the folder `adcd/` with `run_info.json`, `run_log.csv` and the maps.
Upload it as a Kaggle dataset called `adcd-results`; notebook 3 finds it by itself.

**Result format (STANDARD):** `results/adcd/<size>/<image set>/<version>/<format>/<page>.png`, an 8-bit greyscale PNG the SAME
size as the page, value = round(255 × probability of "tampered"). Image set = the folder in the manifest path
(`textctrl`, `shared49`, `anytext` or `diffste`).

**How ADCD-Net reads an image (important for the write-up):** the official loader reads the *decoded pixels*, shrinks the long
side to 512 px if it is longer (this is our `shrink512`), **converts the page to greyscale**, and then **always re-saves it as
a JPEG of quality 100 inside the loader** to get the DCT numbers. So PNG and JPG inputs go through the same path; a JPG page
enters as its decoded pixels, and ADCD-Net never sees colour. We do not change this.

## 1. CONFIG

**Why (ELI5):** every setting in one place. `SMOKE=True` runs 3 pages per set so that a mistake costs minutes, not hours.
The notebook also builds a small to-do list from what already exists, so a stopped run can simply be started again.

In [ ]:
# standard tools for files, text, time and shell commands
import os, re, io, sys, json, glob, time, shutil, zipfile, pickle, shlex, subprocess, random, warnings
# the date for the logs
from datetime import datetime
# arrays and tables
import numpy as np
import pandas as pd
# image reading and writing
from PIL import Image
# OpenCV (the same image reader the detectors use)
import cv2

# ======================= CONFIG: change settings here and nowhere else =======================
# which detector this notebook runs (it names the result folder)
DETECTOR = "adcd"
# True = only SMOKE_N pages per set (a quick test); it MUST be False for the real run (Save & Run All)
SMOKE = False
# pages per set in a smoke run
SMOKE_N = 3
# seed for every random choice
SEED = 42
# the sizes to run: shrink512 = long side 512; full = full resolution in 512 x 512 pieces
SIZES = ["shrink512", "full"]
# piece side (the detector's own input size)
PIECE = 512
# images handled per chunk (a chunk is prepared, run, stitched and then deleted, so the disk never fills up)
CHUNK_IMAGES_FULL = 100
# the same for shrink512 (one small image per page, so bigger chunks)
CHUNK_IMAGES_SHRINK = 400
# the notebook stops if its own timed estimate is longer than this many minutes (a smoke run never stops)
MAX_RUN_MINUTES = 150
# how many images a full run has: 100 pages x 5 versions x 2 formats + 49 x 4 x 2 shared + 49 x 2 x 2 AI edits (skipped donors make it smaller)
EXPECTED_IMAGES_FULL_RUN = 1588
# optional: one folder that holds ALL inputs (only used for tests; leave empty on Kaggle)
INPUT_ROOT_OVERRIDE = ""
# ===============================================================================================

# are we on Kaggle?
IN_KAGGLE = os.path.exists("/kaggle/input")
# Kaggle
if IN_KAGGLE:
    # where the datasets are mounted
    SEARCH_ROOTS = ["/kaggle/input"]
    # the folder whose content is saved as notebook output
    WORKING = "/kaggle/working"
    # big temporary files go here (NOT saved as output)
    TMP_ROOT = "/tmp"
# anywhere else (a laptop test)
else:
    # look in the current folder
    SEARCH_ROOTS = [os.getcwd()]
    # outputs next to the notebook
    WORKING = os.path.join(os.getcwd(), "w7_2A_working")
    # temporary files inside the working folder
    TMP_ROOT = os.path.join(WORKING, "tmp")
# an explicit input folder wins
if INPUT_ROOT_OVERRIDE:
    # use it
    SEARCH_ROOTS = [INPUT_ROOT_OVERRIDE]
# the notebook name used in zip names
NB_TAG = "2A"
# the folder that is sent on: results/<detector>/ (maps + run_info.json + run_log.csv)
RES_DIR = os.path.join(WORKING, "results", DETECTOR)
# logs, config, timing, figures of this notebook
RUN_DIR = os.path.join(WORKING, "w7_" + NB_TAG + "_run")
# big temporary files (pieces, probability maps)
WORK_DIR = os.path.join(TMP_ROOT, "w7_" + NB_TAG + "_work")
# the folder for our own helper scripts (run with the detector's venv Python)
SCRIPTS_DIR = os.path.join(WORKING, "adcd_scripts")
# make every folder
for d in [RES_DIR, RUN_DIR, WORK_DIR, SCRIPTS_DIR]:
    # create it
    os.makedirs(d, exist_ok=True)
# seeds
random.seed(SEED)
# numpy seed
np.random.seed(SEED)

# how many GPUs does this machine have? (asked from the driver, so no torch is needed in this kernel)
try:
    # one line per GPU
    gpu_lines = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
# no nvidia-smi
except Exception:
    # no GPU lines
    gpu_lines = []
# at least 1 (a laptop test uses a stub in place of the GPU)
N_GPUS = max(1, len(gpu_lines))
# show the GPUs
print("GPUs:", gpu_lines if gpu_lines else "none found (set Accelerator = GPU T4 x2 on Kaggle)")

# the settings as a dictionary (saved as config.json; the notebook 06 cells below also read from it)
CONFIG = {"detector": DETECTOR, "SMOKE": SMOKE, "SMOKE_N": SMOKE_N, "seed": SEED, "sizes": SIZES, "piece": PIECE, "threshold": 0.5,
          "chunk_images_full": CHUNK_IMAGES_FULL, "chunk_images_shrink": CHUNK_IMAGES_SHRINK, "max_run_minutes": MAX_RUN_MINUTES,
          "n_gpus": N_GPUS, "started": datetime.now().isoformat(timespec="seconds")}

# versions and the repo, as in notebook 06 (the cells copied from it read these names)
CONFIG.update({"python_version": "3.10", "torch_version": "2.3.0+cu121", "albumentations_version": "2.0.8",
               "repo_url": "https://github.com/KAHIMWONG/ACDC-Net"})
# where the ADCD-Net code is cloned
REPO_DIR = os.path.join(WORKING, "ADCD-Net-code")
# the Python 3.10 venv lives in /tmp so its ~5 GB is NOT saved as notebook output
VENV_DIR = "/tmp/venv310"
# the venv's Python (we run ADCD-Net scripts with this one)
VENV_PY = VENV_DIR + "/bin/python"
# folder for the path lists that infer.py reads, and for per-image score tables
PKL_DIR = os.path.join(WORK_DIR, "path_pkl")
# score tables of infer.py (small csv files, kept)
SCORES_DIR = os.path.join(RUN_DIR, "scores")
# make them
for d in [PKL_DIR, SCORES_DIR]:
    # create
    os.makedirs(d, exist_ok=True)

# save config.json
with open(os.path.join(RUN_DIR, "config.json"), "w") as f:
    # write it
    json.dump(CONFIG, f, indent=2)

# the list of checks (name, value, passed, hard)
CHECKS = []

# record one check and print it
def check(name, value, passed, hard=True):
    # store it
    CHECKS.append(dict(check=name, value=str(value), passed=bool(passed), hard=hard))
    # show it
    print(("PASS " if passed else ("FAIL " if hard else "WARN ")) + name + " -> " + str(value))

# stop with a clear message if a hard check failed
def stop_if_failed():
    # the hard failures
    failed = [c["check"] for c in CHECKS if c["hard"] and not c["passed"]]
    # stop
    if failed:
        raise RuntimeError("STOP: these checks failed: " + "; ".join(failed))

# a loud warning in smoke mode
if SMOKE:
    # banner
    print("#" * 78)
    print("#  SMOKE=True -> only", SMOKE_N, "pages per set. Set SMOKE=False before Save & Run All.")
    print("#" * 78)
# show the settings
print({k: v for k, v in CONFIG.items() if not isinstance(v, dict)})

## 2. Find the inputs (no typing)

**Why (ELI5):** Kaggle puts each attached dataset in its own folder, sometimes nested one level deeper. We look for a file we
know is inside each one: `manifest.csv` (the page sets), `ADCDNet.pth` and `docres.pkl` (the model), `qt_table.pk` (JPEG
tables, from the Week 5 run) and a folder `orig/ocr` (the Week 5 OCR masks of the 100 untouched TextCtrl pages).
Then the manifest is turned into a list of unique images, with the page size read from each file header.

In [ ]:
# folders we never walk into (they hold thousands of images or are our own outputs)
SKIP_DIRS = {"images", "masks", "figures", "cache", "results", "shrink512", "full", "default", "eval_outputs", "w7_2A_working", "w7_2B_working"}

# list every folder (up to max_depth levels down) that contains a file called `marker`
def find_folders_with(marker, max_depth=6):
    # the answer
    found = []
    # look under each search root
    for root in SEARCH_ROOTS:
        # skip roots that do not exist
        if not os.path.isdir(root):
            continue
        # walk down from the root (follow links: Kaggle mounts datasets in two styles)
        for folder, subfolders, files in os.walk(root, followlinks=True):
            # how many levels below the root
            depth = folder[len(root):].count(os.sep)
            # the marker is here
            if marker in files:
                found.append(folder)
            # do not enter big folders or go too deep
            subfolders[:] = [] if depth >= max_depth else [d for d in subfolders if d not in SKIP_DIRS]
    # give back the folders
    return found

# the first folder that holds `marker`, or a clear message (name = what to attach on Kaggle)
def need_folder(marker, name):
    # search
    hits = find_folders_with(marker)
    # nothing
    if not hits:
        raise FileNotFoundError("Cannot find " + marker + ". Attach the dataset " + name + " (Add Input). Looked under: " + str(SEARCH_ROOTS))
    # first hit
    return sorted(hits)[0]

# ---------- the dataset PATH_TO_W7_COMMON_SETS_DATASET ----------
# every folder with a manifest.csv; prefer the one that also holds masks/
cands = find_folders_with("manifest.csv")
# nothing found: stop with a clear message
if not cands:
    raise FileNotFoundError("No manifest.csv found. Attach the dataset PATH_TO_W7_COMMON_SETS_DATASET. Looked under: " + str(SEARCH_ROOTS))
# the real dataset has a masks/ folder next to the manifest
cands.sort(key=lambda p: (not os.path.isdir(os.path.join(p, "masks")), p))
# the dataset folder (manifest paths are relative to it)
SETS_DIR = cands[0]
# read the manifest, every column as text
manifest = pd.read_csv(os.path.join(SETS_DIR, "manifest.csv"), dtype=str, keep_default_na=False)
# the three page sets
SETS = ["textctrl", "anytext", "diffste"]
# SMOKE: keep the first SMOKE_N pages of each set
if SMOKE:
    # pages to keep per set
    keep = {s: sorted(manifest[manifest["set"] == s]["page"].unique())[:SMOKE_N] for s in SETS}
    # filter the rows
    manifest = manifest[[p in keep[s] for s, p in zip(manifest["set"], manifest["page"])]].reset_index(drop=True)
# split the path: images/<image set>/<version>/<format>/<page>.<ext>
parts = manifest["path"].str.split("/", expand=True)
# the folder set the image lives in (textctrl, shared49, anytext or diffste)
manifest["img_set"] = parts[1]
# the file stem (= page id)
manifest["stem"] = parts[4].str.rsplit(".", n=1).str[0]
# one row per unique image file (anytext and diffste rows share the shared49 files)
images = manifest.drop_duplicates("path").reset_index(drop=True)
# full path of each image
images["abs_path"] = [os.path.join(SETS_DIR, p) for p in images["path"]]
# full path of each box mask
images["abs_mask"] = [os.path.join(SETS_DIR, p) for p in images["mask_path"]]
# page sizes, read from the file headers only (width, height)
wh = [Image.open(p).size for p in images["abs_path"]]
# width column
images["W"] = [s[0] for s in wh]
# height column
images["H"] = [s[1] for s in wh]
# a running number (used to name temporary files)
images["idx"] = range(len(images))
# print what we found
print("dataset folder :", SETS_DIR)
print("manifest rows  :", len(manifest), "| unique image files:", len(images))
print(images.groupby(["img_set", "version", "format"]).size().unstack(fill_value=0).to_string())
# the pieces a full-size image needs (rows x columns of 512 x 512, padded at the right and bottom)
images["n_rows"] = np.ceil(images["H"] / PIECE).astype(int)
# columns
images["n_cols"] = np.ceil(images["W"] / PIECE).astype(int)
# pieces per image
images["n_pieces"] = images["n_rows"] * images["n_cols"]
# totals
print("images:", len(images), "| full-size pieces in total:", int(images["n_pieces"].sum()), "| mean pieces per image: %.1f" % images["n_pieces"].mean())

# where a result map lives: results/<detector>/<size>/<image set>/<version>/<format>/<page>.png
def result_path(size, row):
    # follow the standard layout of CLAUDE.md
    return os.path.join(RES_DIR, size, row["img_set"], row["version"], row["format"], row["stem"] + ".png")

In [ ]:
# ---------- ADCD-Net inputs ----------
# the weights: the folder that holds ADCDNet.pth
CKPT_PATH = os.path.join(need_folder("ADCDNet.pth", "PATH_TO_ADCD_WEIGHTS_DATASET"), "ADCDNet.pth")
# the DocRes backbone weights
DOCRES_PATH = os.path.join(need_folder("docres.pkl", "PATH_TO_ADCD_WEIGHTS_DATASET"), "docres.pkl")
# the JPEG quantisation tables built in Week 5
QT_PATH = os.path.join(need_folder("qt_table.pk", "PATH_TO_ADCD_RUN_OUTPUTS_DATASET"), "qt_table.pk")

# find a folder called `name` whose parent folder is called `parent` (the Week 5 OCR masks of the untouched pages: orig/ocr)
def find_dir_named(name, parent):
    # the answer
    hits = []
    # look under each search root
    for root in SEARCH_ROOTS:
        # skip roots that do not exist
        if not os.path.isdir(root):
            continue
        # walk down
        for folder, subfolders, files in os.walk(root, followlinks=True):
            # depth below the root
            depth = folder[len(root):].count(os.sep)
            # this is the folder we want
            if os.path.basename(folder) == name and os.path.basename(os.path.dirname(folder)) == parent:
                hits.append(folder)
            # do not go too deep or into the image folders of PATH_TO_W7_COMMON_SETS_DATASET
            subfolders[:] = [] if depth >= 7 else [d for d in subfolders if d not in SKIP_DIRS]
    # first hit
    return sorted(hits)[0] if hits else None

# the Week 5 OCR masks of the untouched TextCtrl pages
TC_OCR_DIR = find_dir_named("ocr", "orig")
# stop with a clear message if they are missing
if TC_OCR_DIR is None:
    raise FileNotFoundError("Cannot find the folder inputs/orig/ocr (Week 5 OCR masks). Attach the dataset PATH_TO_ADCD_RUN_OUTPUTS_DATASET.")
# the TextCtrl pages of this run need one mask each
tc_pages = sorted(images[images["img_set"] == "textctrl"]["page"].unique())
# which are missing
tc_missing = [p for p in tc_pages if not os.path.exists(os.path.join(TC_OCR_DIR, p + ".png"))]
# stop if any are missing
if tc_missing:
    raise FileNotFoundError("Week 5 OCR masks missing for " + str(len(tc_missing)) + " pages, e.g. " + str(tc_missing[:5]) + " in " + TC_OCR_DIR)
# show what we found
print("ADCDNet.pth :", CKPT_PATH)
print("docres.pkl  :", DOCRES_PATH)
print("qt_table.pk :", QT_PATH)
print("TextCtrl OCR masks (Week 5):", TC_OCR_DIR, "|", len(tc_pages), "pages in this run")

## 3. The ADCD-Net environment (copied unchanged from Week 7 notebook 06, cells R2.1–R2.7)

**Why (ELI5):** ADCD-Net was written for Python 3.10 and torch 2.3, which Kaggle does not have. We build a small separate
Python 3.10 environment in `/tmp` (about 5–8 minutes the first time), install the packages one at a time, compile
`jpeg2dct`, clone the model code and check that everything imports. These cells are the ones that already worked in
notebook 06; only the folder names come from section 1.

In [ ]:
# [ENV]
# Function: run one shell command, show a short log, stop with the last 30 lines if it fails
def run(cmd, show_tail=3):
    # Start a stopwatch
    t0 = time.time()
    # Run the command in a shell and capture everything it prints
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    # Join normal output and error output, split into lines
    lines = (res.stdout + res.stderr).strip().splitlines()
    # Seconds the command took
    secs = time.time() - t0
    # If it failed: print the last 30 lines and stop the cell
    if res.returncode != 0:
        print("\n".join(lines[-30:]))
        raise RuntimeError(f"FAILED after {secs:.0f}s: {cmd}")
    # If it worked: print the last few lines and the time
    if show_tail:
        print("\n".join(lines[-show_tail:]))
    print(f"OK ({secs:.0f}s): {cmd[:100]}")

# Keep uv's download cache in /tmp (not saved as notebook output)
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"
# Copy files instead of linking them (avoids a warning between different disks)
os.environ["UV_LINK_MODE"] = "copy"

# Install uv into Kaggle's Python
run("pip install -q uv", show_tail=0)
# Build the venv only if it is not there yet (resumable)
if not os.path.exists(VENV_PY):
    # uv downloads its own Python 3.10 and adds pip/setuptools/wheel (--seed)
    run(f"uv venv {VENV_DIR} --python {CONFIG['python_version']} --seed")
# Print the venv's Python version
run(f"{VENV_PY} --version", show_tail=1)

In [ ]:
# [ENV]
# The constraints file: versions every later install must respect
CONSTRAINTS = "/tmp/constraints.txt"
with open(CONSTRAINTS, "w") as f:
    f.write("torch==2.3.0+cu121\ntorchvision==0.18.0+cu121\nnumpy<2\n")

# Install torch + torchvision (ds.py imports torchvision) from PyTorch's CUDA 12.1 index
run(f"uv pip install --python {VENV_PY} torch==2.3.0+cu121 torchvision==0.18.0+cu121 "
    f"--index-url https://download.pytorch.org/whl/cu121")
# Ask the venv's Python: torch version, CUDA version, can it see the GPU?
run(f"{VENV_PY} -c \"import torch; print('torch', torch.__version__, '| CUDA', torch.version.cuda, "
    f"'| GPU available:', torch.cuda.is_available())\"", show_tail=1)

In [ ]:
# [ENV]
# Packages the ADCD-Net repo imports (numpy first, so nothing pulls in numpy 2)
PACKAGES = ["numpy<2", "opencv-python-headless", "albumentations==2.0.8", "lmdb", "six",
            "pillow", "tqdm", "einops", "efficientnet_pytorch", "tensorboard"]

# One row per package: name, status, seconds
install_log = []
for pkg in PACKAGES:
    # Stopwatch for this package
    t0 = time.time()
    # Install it into the venv, respecting the constraints file (stops the cell on failure)
    run(f"uv pip install --python {VENV_PY} -c {CONSTRAINTS} \"{pkg}\"", show_tail=0)
    # Record that it worked and how long it took
    install_log.append({"package": pkg, "status": "OK", "seconds": round(time.time() - t0, 1)})

# Show the install table
print(pd.DataFrame(install_log).to_string(index=False))
# Confirm numpy stayed below version 2
run(f"{VENV_PY} -c \"import numpy; print('numpy', numpy.__version__)\"", show_tail=1)

In [ ]:
# [ENV]
# Install the system JPEG library headers and a C/C++ compiler (Kaggle runs as root, so no sudo)
run("apt-get update -qq && apt-get install -y -qq libjpeg-dev build-essential", show_tail=0)
# Build tools the old setup.py needs inside the venv
run(f"uv pip install --python {VENV_PY} -c {CONSTRAINTS} \"setuptools<70\" wheel", show_tail=0)
# Compile and install jpeg2dct against the venv's numpy
run(f"uv pip install --python {VENV_PY} -c {CONSTRAINTS} --no-build-isolation jpeg2dct")

# Tiny test: save a random 64x64 grey JPEG and read its DCT numbers back
test_code = (
    "import numpy as np, tempfile; from PIL import Image; from jpeg2dct.numpy import load; "
    "p = tempfile.mktemp(suffix='.jpg'); "
    "Image.fromarray((np.random.rand(64, 64) * 255).astype('uint8')).save(p, quality=100); "
    "y, _, _ = load(p, normalized=False); print('jpeg2dct OK: DCT shape', y.shape)"
)
# Run the test with the venv's Python (64 px / 8 = 8 blocks per side, 64 numbers per block)
run(f"{VENV_PY} -c \"{test_code}\"", show_tail=1)

In [ ]:
# [ENV]
# Clone only the latest version (depth 1), and only if not cloned already (resumable)
if not os.path.isdir(REPO_DIR):
    run(f"git clone -q --depth 1 {CONFIG['repo_url']} {REPO_DIR}", show_tail=0)
# Read the exact commit we got
commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Repo commit:", commit)
print("Root files :", sorted(os.listdir(REPO_DIR)))

# Save the commit in CONFIG and re-save config.json
CONFIG["repo_commit"] = commit
with open(f"{RUN_DIR}/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

In [ ]:
%%writefile /kaggle/working/adcd_scripts/check_env.py
# check_env.py: run with the venv Python from inside the repo folder.
# Usage: python check_env.py <path to ADCDNet.pth> <path to docres.pkl>

# Standard tools: command-line arguments and Python version
import sys, platform
# Read the two weight-file paths from the command line
ckpt_path, docres_path = sys.argv[1], sys.argv[2]

# Import the main libraries and print their versions
import torch, numpy, cv2, albumentations
print("python", platform.python_version(), "| torch", torch.__version__, "| CUDA available", torch.cuda.is_available())
# jpeg2dct has no version number to print, so just show it imports
from jpeg2dct.numpy import load
print("numpy", numpy.__version__, "| cv2", cv2.__version__, "| albumentations", albumentations.__version__, "| jpeg2dct ok")

# Make the repo folder importable (we are running from inside it)
sys.path.insert(0, ".")
# Import the repo's settings and point them at our weight files (in memory only, cfg.py is not edited)
import cfg
cfg.ckpt = ckpt_path
cfg.docres_ckpt_path = docres_path
# Import the data module and the model class
import ds
print("import ds OK", end=" | ")
from model.model import ADCDNet
print("import ADCDNet OK")

# Open the ADCD-Net checkpoint the same way main.py does (weights_only=True = safe loading)
ck = torch.load(ckpt_path, map_location="cpu", weights_only=True)
# Print only the key names and how many tensors are stored under 'model'
print("ADCDNet.pth top-level keys:", list(ck.keys())[:10], "| tensors in 'model':", len(ck["model"]))
# Same for the DocRes backbone, the way model.py loads it
dr = torch.load(docres_path, map_location="cpu", weights_only=True)
print("docres.pkl  top-level keys:", list(dr.keys())[:10], "| tensors in 'model_state':", len(dr["model_state"]))
print("ALL CHECKS PASSED")

In [ ]:
# [ENV]
# Run the check script with the venv Python, from inside the repo folder, and capture its output
res = subprocess.run([VENV_PY, f"{SCRIPTS_DIR}/check_env.py", CKPT_PATH, DOCRES_PATH],
                     cwd=REPO_DIR, capture_output=True, text=True)
# Keep normal output plus only the last 30 lines of any errors/warnings
report = res.stdout + "\n".join(res.stderr.strip().splitlines()[-30:])
# Show it
print(report)
# Save it next to config.json so the versions are on record
with open(f"{RUN_DIR}/env_versions.txt", "w") as f:
    f.write(report)
print("Saved:", f"{RUN_DIR}/env_versions.txt", "| exit code:", res.returncode, "(0 = success)")
# Stop the notebook if the check failed (no point running inference with a broken environment)
if res.returncode != 0:
    raise RuntimeError("check_env.py failed: read the last lines printed above")

**`infer.py`** (copied unchanged from notebook 06 / notebook 04): runs ADCD-Net on a list of (image, box mask, OCR mask)
triples with the official loader and saves one probability map (`.npy`) per image. Only its inputs change.

In [ ]:
%%writefile /kaggle/working/adcd_scripts/infer.py
# infer.py: run pretrained ADCD-Net on one path list and save tamper-probability maps.
# Run with the venv Python. One GPU, no DDP. Copies the evaluation steps of the repo's main.py.

# Standard tools
import argparse, os, sys, time, csv
import numpy as np
import torch

# Read the command-line options
ap = argparse.ArgumentParser()
# Folder of the ADCD-Net repo
ap.add_argument("--repo")
# Folder with the <ds_name>.pkl path lists, and which list to run
ap.add_argument("--pkl_dir")
ap.add_argument("--ds_name")
# Where to save the .npy maps and the per-image CSV
ap.add_argument("--out_dir")
ap.add_argument("--csv")
# Weight files and the quantization-table file
ap.add_argument("--ckpt")
ap.add_argument("--docres")
ap.add_argument("--qt")
# Longest side after shrinking (official: 512), max images to do (0 = all), seed
ap.add_argument("--max_size", type=int, default=512)
ap.add_argument("--limit", type=int, default=0)
ap.add_argument("--seed", type=int, default=42)
args = ap.parse_args()

# Work from inside the repo so its own imports (cfg, ds, model) work
os.chdir(args.repo)
sys.path.insert(0, args.repo)
# Point the repo's settings at our files (in memory only; cfg.py is not edited)
import cfg
cfg.ckpt = args.ckpt
cfg.docres_ckpt_path = args.docres
cfg.pkl_dir = args.pkl_dir
cfg.qt_path = args.qt
cfg.val_max_size = args.max_size
# The repo's data loader, its padding function, and the model class
from ds import GeneralValDs, pad_collate
from model.model import ADCDNet
# float16 mode, exactly as main.py uses in evaluation
from torch.amp import autocast
from torch.utils.data import DataLoader, Subset

# Seeds (the evaluation itself has no randomness, but this is our rule)
torch.manual_seed(args.seed)
np.random.seed(args.seed)
# Small number to avoid dividing by zero (same as main.py)
EPS = 1e-8

# Build the dataset from the path list
ds = GeneralValDs(args.ds_name)
# Image names exactly as the loader makes them (file name up to the first dot)
names = [os.path.basename(p[0]).split(".")[0] for p in ds.path_list]
# Make the output folder
os.makedirs(args.out_dir, exist_ok=True)
# Only the images without a saved map (resumable)
todo = [i for i, n in enumerate(names) if not os.path.exists(f"{args.out_dir}/{n}.npy")]
# Apply the limit if one was given
if args.limit > 0:
    todo = todo[:args.limit]
print(f"{len(names)} images in {args.ds_name}, {len(todo)} to do", flush=True)
# Nothing to do: stop here
if not todo:
    sys.exit(0)
# One image at a time, padded the official way
dl = DataLoader(Subset(ds, todo), batch_size=1, shuffle=False, num_workers=0, collate_fn=pad_collate)

# Build the model (this also loads docres.pkl) and move it to the GPU
t0 = time.time()
model = ADCDNet().cuda()
# Load the ADCD-Net weights the way main.py does (remove the 'module.' prefix left by multi-GPU training)
state = torch.load(args.ckpt, map_location="cpu", weights_only=True)["model"]
state = {k.replace("module.", ""): v for k, v in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
# Evaluation mode (no training behaviour)
model.eval()
print(f"model ready in {time.time() - t0:.1f}s | missing keys: {len(missing)} | unexpected keys: {len(unexpected)}", flush=True)
# Show a few key names if anything did not match
if missing or unexpected:
    print("  first missing:", missing[:5], "| first unexpected:", unexpected[:5], flush=True)

# Write the CSV header the first time
cols = ["image", "h", "w", "seconds", "max_prob", "p999", "top01_mean", "mean_prob",
        "precision", "recall", "f1", "gt_pixels", "pred_pixels"]
if not os.path.exists(args.csv):
    with open(args.csv, "w", newline="") as f:
        csv.writer(f).writerow(cols)

# No gradients needed for prediction
with torch.no_grad():
    # Stopwatch (includes loading + preprocessing of each image)
    t_last = time.time()
    for k, items in enumerate(dl, 1):
        # The 5 tensors the model needs, on the GPU (the true mask is passed but not used in eval mode)
        img, dct, qt, mask, ocr_mask = [x.cuda() for x in items[:5]]
        # Image name and its real size before padding
        name = items[5][0]
        h, w = items[6][0].tolist()
        # Run the model in float16 and keep the first output (the 2-class scores)
        with autocast(device_type="cuda", dtype=torch.float16):
            logits = model(img, dct, qt, mask, ocr_mask, is_train=False)[0]
        # Crop away the padding, back to float32
        logit = logits[0, :, :h, :w].float()
        # Probability of class 1 ("tampered") per pixel
        prob = torch.softmax(logit, dim=0)[1]
        # Official prediction = the class with the higher score (same as probability >= 0.5)
        pred = logit.argmax(0)
        # The answer key at the same size (1 = tampered)
        y = (mask[0, 0, :h, :w] == 1).long()
        # Official-style precision, recall and F1 for this image (main.py compute_f1)
        matched = (pred * y).sum().item()
        p = matched / (pred.sum().item() + EPS)
        r = matched / (y.sum().item() + EPS)
        f1 = 2 * p * r / (p + r + EPS)
        # Save the probability map as float16
        prob_np = prob.cpu().numpy()
        np.save(f"{args.out_dir}/{name}.npy", prob_np.astype(np.float16))
        # Image-level scores: max, 99.9th percentile, mean of the top 0.1 % of pixels, overall mean
        flat = np.sort(prob_np.ravel())
        n_top = max(1, int(np.ceil(0.001 * flat.size)))
        top01 = float(flat[-n_top:].mean())
        # Seconds for this image
        torch.cuda.synchronize()
        secs = time.time() - t_last
        t_last = time.time()
        # Append the row to the CSV
        row = [name, h, w, round(secs, 3), float(flat[-1]), float(np.percentile(flat, 99.9)), top01,
               float(flat.mean()), p, r, f1, int(y.sum().item()), int(pred.sum().item())]
        with open(args.csv, "a", newline="") as f:
            csv.writer(f).writerow(row)
        # Progress: first 3 images, then every 25th, and the last one
        if k <= 3 or k % 25 == 0 or k == len(todo):
            print(f"[{k}/{len(todo)}] {name}  {secs:.2f}s  max prob {flat[-1]:.3f}  F1 {f1:.3f}", flush=True)
print("done", flush=True)

In [ ]:
# [ENV]
# Ask the venv's Python to compile infer.py without running it (catches copy/paste errors)
run(f"{VENV_PY} -m py_compile {SCRIPTS_DIR}/infer.py", show_tail=0)
# Count the lines of the script, to compare with notebook 04
n_lines = len(open(f"{SCRIPTS_DIR}/infer.py").read().splitlines())
print(f"infer.py compiles OK ({n_lines} lines) at {SCRIPTS_DIR}/infer.py")

## 4. OCR text masks: Week 5 masks for TextCtrl, PaddleOCR for the 49 shared pages

**Why (ELI5):** ADCD-Net is told *where the text is* (a white mask over every text line) so it can compare text pixels with
clean background. The authors make this mask with PaddleOCR (a text *finder*, it does not read the letters).

- **TextCtrl pages:** we reuse the Week 5 masks of the 100 untouched pages.
- **The 49 shared pages:** we run PaddleOCR (Week 5 recipe, on the CPU in its own small environment, about 7 minutes) on the
  **untouched** page. The same mask is used for all five versions of a page, because only one word differs.
- The mask must have the page's exact size (the loader shrinks image and mask together); any mismatch is resized with
  nearest-neighbour and counted.

In [ ]:
# the shared pages of this run (AnyText and DiffSTE use the same 49)
shared_pages = sorted(manifest[manifest["set"].isin(["anytext", "diffste"])]["page"].unique())
# the untouched PNG of each shared page (that is the image PaddleOCR reads)
unt = images[(images["img_set"] == "shared49") & (images["version"] == "untouched") & (images["format"] == "png")].set_index("page")
# every shared page must have one
assert all(p in unt.index for p in shared_pages), "an untouched PNG is missing for a shared page"
# where the new masks go (saved in the outputs, zip 1)
OCR_NEW_DIR = os.path.join(WORKING, "ocr_masks_shared49")
# make it
os.makedirs(OCR_NEW_DIR, exist_ok=True)
# the list for the OCR script: [variant, image path, mask path]
ocr_manifest = [["shared49", unt.loc[p, "abs_path"], os.path.join(OCR_NEW_DIR, p + ".png")] for p in shared_pages]
# where the list is saved
OCR_MANIFEST = os.path.join(WORK_DIR, "ocr_manifest.json")
# save it
with open(OCR_MANIFEST, "w") as f:
    # write
    json.dump(ocr_manifest, f)
# the OCR timing file
OCR_TIMING = os.path.join(RUN_DIR, "ocr_timing.csv")
# these folders and files are added to the zips
EXTRA_ALL_DIRS = [(OCR_NEW_DIR, "ocr_masks_shared49")]
# the OCR timing is a table without statement text
EXTRA_SAFE_FILES = ["ocr_timing.csv"]
# show
print(len(ocr_manifest), "shared pages need an OCR mask;", sum(os.path.exists(m[2]) for m in ocr_manifest), "already exist")

In [ ]:
# [ENV]
# Where the OCR environment lives (in /tmp, so it is not saved as notebook output)
OCR_VENV = "/tmp/venv_ocr"
# Its Python interpreter
OCR_PY = f"{OCR_VENV}/bin/python"

# Create the venv only if it does not exist yet (resumable)
if not os.path.exists(OCR_PY):
    run(f"uv venv {OCR_VENV} --python {CONFIG['python_version']} --seed")
# Install the packages one at a time, numpy first so nothing pulls numpy 2
for pkg in ["numpy<2", "paddlepaddle", "paddleocr"]:
    run(f"uv pip install --python {OCR_PY} \"{pkg}\"", show_tail=0)

# Ask the OCR venv for the versions it ended up with
ocr_check = ("import paddle, paddleocr, cv2; "
         "print('paddle', paddle.__version__, '| paddleocr', paddleocr.__version__, '| cv2', cv2.__version__)")
run(f"{OCR_PY} -c \"{ocr_check}\"", show_tail=1)

**The OCR script** (copied unchanged from Week 5 notebook 04, cell 32): PaddleOCR PP-OCRv5 server text detector, polygons with score > 0.5 filled white.

In [ ]:
%%writefile /kaggle/working/adcd_scripts/ocr_masks.py
# ocr_masks.py: make PaddleOCR text masks, one PNG per image. Run with the OCR venv's Python.
# Usage: python ocr_masks.py <manifest.json> <timing.csv> <limit (0 = all)>

# Standard tools
import sys, os, json, time, csv
# Stop PaddleOCR from checking model-hosting sites at start-up (saves seconds)
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
# Image tools and the PaddleOCR text detector
import cv2
import numpy as np
from paddleocr import TextDetection

# Read the command-line arguments
manifest_path, timing_csv, limit = sys.argv[1], sys.argv[2], int(sys.argv[3])
# The manifest is a list of [variant, image_path, mask_output_path]
items = json.load(open(manifest_path))
# Keep only the masks that do not exist yet (this makes the script resumable)
todo = [it for it in items if not os.path.exists(it[2])]
print(f"{len(items)} images in manifest, {len(todo)} masks still to make", flush=True)
# If a limit was given, only do that many (used for the timing test)
if limit > 0:
    todo = todo[:limit]

# Load the text detector (same model name as the repo's seg_char.py).
# enable_mkldnn=False switches off Intel's oneDNN speed-up: in this Paddle version it crashes
# on CPU (ConvertPirAttribute2RuntimeAttribute error). Results are the same, only slower.
t0 = time.time()
model = TextDetection(model_name="PP-OCRv5_server_det", enable_mkldnn=False)
print(f"model loaded in {time.time() - t0:.1f}s", flush=True)

# Write the timing-file header the first time
if not os.path.exists(timing_csv):
    with open(timing_csv, "w", newline="") as f:
        csv.writer(f).writerow(["variant", "image", "seconds", "n_text_regions"])

# Go through the images one by one
for i, (variant, img_path, out_path) in enumerate(todo, 1):
    # Stopwatch for this page
    t = time.time()
    # Detect text regions (batch of 1, like the repo)
    res = model.predict(input=img_path, batch_size=1)[0]
    # Polygons around the text and the confidence of each polygon
    polys, scores = res["dt_polys"], res["dt_scores"]
    # Read the page only to get its height and width
    h, w = cv2.imread(img_path).shape[:2]
    # Start with an all-black mask
    mask = np.zeros((h, w), dtype=np.uint8)
    # Number of polygons kept
    kept = 0
    # Fill every polygon whose score is above 0.5 (repo rule)
    for poly, score in zip(polys, scores):
        if score > 0.5:
            cv2.fillPoly(mask, [poly.astype(np.int32)], 1)
            kept += 1
    # Encode as PNG (0/255), even if the file name ends in .jpg
    ok, buf = cv2.imencode(".png", mask * 255)
    # Write to a temporary name, then rename: no half-written files after a crash
    with open(out_path + ".tmp", "wb") as f:
        f.write(buf.tobytes())
    os.replace(out_path + ".tmp", out_path)
    # Seconds this page took
    secs = time.time() - t
    # Log it
    with open(timing_csv, "a", newline="") as f:
        csv.writer(f).writerow([variant, os.path.basename(img_path), round(secs, 2), kept])
    # Print progress for the first 3 pages, then every 10th
    if i <= 3 or i % 10 == 0 or i == len(todo):
        print(f"[{i}/{len(todo)}] {variant}/{os.path.basename(img_path)}  {secs:.1f}s  regions kept: {kept}", flush=True)
print("done", flush=True)

In [ ]:
# a command that prints its output as it arrives (copied from notebook 06)
def run_stream(cmd_list, env=None):
    # start the process, merging error text into normal text
    proc = subprocess.Popen(cmd_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    # keep every line so the last 30 can be shown on failure
    lines = []
    # print each line the moment it arrives
    for line in proc.stdout:
        # show it
        print(line, end="")
        # keep it
        lines.append(line)
    # wait for the process to end
    proc.wait()
    # if it failed, show the last 30 lines and stop
    if proc.returncode != 0:
        # show them
        print("".join(lines[-30:]))
        # stop the notebook
        raise RuntimeError("Script failed (see the last 30 lines above)")

# masks still to make
remaining = [m for m in ocr_manifest if not os.path.exists(m[2])]
# nothing to do?
if not remaining:
    # say so
    print("All OCR masks already exist.")
# something to do
else:
    # the longest OCR run we accept without asking (minutes)
    MAX_OCR_MINUTES = 30
    # timing test on 3 pages (also downloads the model once)
    t0 = time.time()
    # run it
    run_stream([OCR_PY, SCRIPTS_DIR + "/ocr_masks.py", OCR_MANIFEST, OCR_TIMING, "3"])
    # median seconds per page from the timing file
    sec_per_page = float(pd.read_csv(OCR_TIMING)["seconds"].median())
    # masks still missing now
    left = sum(1 for m in ocr_manifest if not os.path.exists(m[2]))
    # estimate in minutes
    est_ocr = sec_per_page * left / 60
    # show it
    print("OCR estimate: %d masks left at %.1f s/page = %.1f min" % (left, sec_per_page, est_ocr))
    # too long: stop with a message
    if est_ocr > MAX_OCR_MINUTES:
        raise RuntimeError("STOP: OCR estimate %.0f min is over %d min." % (est_ocr, MAX_OCR_MINUTES))
    # run the rest
    run_stream([OCR_PY, SCRIPTS_DIR + "/ocr_masks.py", OCR_MANIFEST, OCR_TIMING, "0"])
    # record the time
    TIMING = [dict(step="PaddleOCR masks (shared pages)", seconds=round(time.time() - t0, 1), note=str(len(ocr_manifest)) + " pages, CPU")]

# every mask must exist now
n_have = sum(os.path.exists(m[2]) for m in ocr_manifest)
# hard check (the helper functions for checks are defined in section 1)
check("OCR masks exist for every shared page", str(n_have) + " of " + str(len(ocr_manifest)), n_have == len(ocr_manifest))
# stop if not
stop_if_failed()

# ---------- one OCR mask per page, with the page's exact size ----------
# page -> mask path
OCR_MASKS = {}
# resized masks go here
os.makedirs(os.path.join(WORK_DIR, "ocr_resized"), exist_ok=True)
# how many masks had to be resized
n_resized = 0
# each page (unique)
for page, g in images.drop_duplicates("page").set_index("page").iterrows():
    # the right mask file: Week 5 for TextCtrl, PaddleOCR (ours) for the shared pages
    mp = os.path.join(TC_OCR_DIR, page + ".png") if g["img_set"] == "textctrl" else os.path.join(OCR_NEW_DIR, page + ".png")
    # its size
    mw, mh = Image.open(mp).size
    # a different size than the page: resize with nearest neighbour (stays 0 / 255)
    if (mw, mh) != (g["W"], g["H"]):
        # read it
        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        # resize to (width, height)
        m = cv2.resize(m, (int(g["W"]), int(g["H"])), interpolation=cv2.INTER_NEAREST)
        # save the resized copy
        mp = os.path.join(WORK_DIR, "ocr_resized", page + ".png")
        # write
        cv2.imwrite(mp, m)
        # count it
        n_resized += 1
    # remember
    OCR_MASKS[page] = mp
# report
check("OCR masks resized to the page size", str(n_resized) + " of " + str(len(OCR_MASKS)) + " pages (0 is expected)", n_resized == 0, hard=False)
# the text coverage of a few masks (a quick look: share of white pixels)
cov = []
# the first 6 pages
for page in list(OCR_MASKS)[:6]:
    # read
    m = cv2.imread(OCR_MASKS[page], cv2.IMREAD_GRAYSCALE)
    # share of text pixels
    cov.append((page, round(float((m > 0).mean()), 3)))
# show
print("share of text pixels in the OCR mask (first pages):", cov)

## 5. Helpers: pieces, path lists, and running `infer.py` on both GPUs

**Why (ELI5):** `infer.py` reads a *list of file paths* and writes one probability map per file. So for every image we prepare
the files the loader should see, hand them to two copies of `infer.py` (one per GPU), and then put the maps back together.

- **shrink512:** the loader itself shrinks a page whose long side is above 512 to 512. We give it the page file (through a
  link with a short unique name); the map comes back at the shrunk size and is resized back to the page size (bilinear).
- **full:** the page is padded with white at the right and bottom up to a multiple of 512 and cut into 512 × 512 pieces
  starting at (0, 0), so the 8 × 8 JPEG blocks stay aligned. Pieces are saved as PNG from the decoded pixels. The OCR mask is
  cut on the **same grid**. The box mask is not needed for prediction, so every piece gets one all-zero mask.
- The work is done in **chunks** (about 100 images): prepare, run, stitch, save the final maps, delete the pieces. The disk
  therefore stays small and a stopped run resumes from the images that have no map yet.

In [ ]:
# the run log: one row per image and size (seconds, max value, status)
RUN_LOG = os.path.join(RES_DIR, "run_log.csv")
# write the header once
if not os.path.exists(RUN_LOG):
    # header line
    with open(RUN_LOG, "w") as f:
        f.write("image,size,set,version,format,page,seconds,max_value,status" + chr(10))

# turn a probability map (0..1) into the 8-bit result: round(255 x probability)
def to8(prob):
    # round, clip to 0..255, store as 8-bit
    return np.clip(np.rint(np.asarray(prob, dtype=np.float32) * 255.0), 0, 255).astype(np.uint8)

# save one finished 8-bit map in the standard layout and add its run-log row
def save_result(size, row, map8, seconds, status="ok"):
    # the file path
    out = result_path(size, row)
    # make its folder
    os.makedirs(os.path.dirname(out), exist_ok=True)
    # the map must have exactly the page size
    assert map8.shape == (row["H"], row["W"]), "map size " + str(map8.shape) + " != page size " + str((row["H"], row["W"])) + " for " + row["path"]
    # write to a temporary name first, then rename (a crash never leaves a half-written map)
    tmp = out[:-4] + "_tmp.png"
    # write the 8-bit PNG
    cv2.imwrite(tmp, map8)
    # rename
    os.replace(tmp, out)
    # one line in the run log
    with open(RUN_LOG, "a") as f:
        f.write(",".join([row["path"], size, row["set"], row["version"], row["format"], row["page"], "%.3f" % seconds, str(int(map8.max())), status]) + chr(10))

# the images that still have no result at this size (this is what makes the run resumable)
def todo_rows(size):
    # the rows
    rows = []
    # check each image
    for r in images.to_dict("records"):
        # no result file yet
        if not os.path.exists(result_path(size, r)):
            # keep it
            rows.append(r)
    # the list
    return rows

# a stopwatch note for the timing table (keep lines made by earlier cells)
TIMING = globals().get("TIMING", [])
# add one line to the timing table
def log_time(step, seconds, note=""):
    # store
    TIMING.append(dict(step=step, seconds=round(seconds, 1), note=note))

# these two lists are filled by the cells that know the extra folders (OCR masks and so on)
EXTRA_ALL_DIRS = globals().get("EXTRA_ALL_DIRS", [])
# the safe extra files
EXTRA_SAFE_FILES = globals().get("EXTRA_SAFE_FILES", [])

In [ ]:
# round up (no import needed)
def ceil_div(a, b):
    # integer ceiling
    return -(-a // b)

# pad an array on the right/bottom up to (h_pad, w_pad) with a fill value
def pad_to(arr, h_pad, w_pad, value):
    # a new array full of the fill value, same number of channels
    out = np.full((h_pad, w_pad) + arr.shape[2:], value, dtype=arr.dtype)
    # copy the original into the top-left corner
    out[:arr.shape[0], :arr.shape[1]] = arr
    # give it back
    return out

# environment that shows only GPU number k to a script
def gpu_env(k):
    # copy the environment and set the visible GPU
    return dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))

# the command that runs infer.py on one path list (same options as notebook 06)
def infer_cmd(ds_name, out_dir, csv_path, limit=0):
    # the list of command words
    return [VENV_PY, SCRIPTS_DIR + "/infer.py", "--repo", REPO_DIR, "--pkl_dir", PKL_DIR, "--ds_name", ds_name,
            "--out_dir", out_dir, "--csv", csv_path, "--ckpt", CKPT_PATH, "--docres", DOCRES_PATH, "--qt", QT_PATH,
            "--max_size", "512", "--limit", str(limit), "--seed", str(SEED)]

# one all-zero 512 x 512 mask shared by every piece (the answer key is not used for prediction)
ZERO_MASK = os.path.join(WORK_DIR, "zero_mask.png")
# write it
cv2.imwrite(ZERO_MASK, np.zeros((PIECE, PIECE), np.uint8))
# folders for pieces, links and probability maps
for d in ["piece_img", "links", "probs", "ocr_pieces"]:
    # create
    os.makedirs(os.path.join(WORK_DIR, d), exist_ok=True)

# OCR mask pieces of one page, cut on the same grid as the image (made once, used by all 10 versions/formats of the page)
def ocr_pieces_for(page, W, H):
    # folder of this page
    d = os.path.join(WORK_DIR, "ocr_pieces", page)
    # rows and columns of pieces
    n_r, n_c = ceil_div(H, PIECE), ceil_div(W, PIECE)
    # the file of each piece
    paths = {(r, c): os.path.join(d, "r%d_c%d.png" % (r, c)) for r in range(n_r) for c in range(n_c)}
    # all there already?
    if not all(os.path.exists(p) for p in paths.values()):
        # make the folder
        os.makedirs(d, exist_ok=True)
        # read the page-size OCR mask
        m = cv2.imread(OCR_MASKS[page], cv2.IMREAD_GRAYSCALE)
        # pad with 0 (no text), like the image is padded with white
        m = pad_to(m, n_r * PIECE, n_c * PIECE, 0)
        # cut and save every piece
        for (r, c), p in paths.items():
            # write the piece
            cv2.imwrite(p, m[r * PIECE:(r + 1) * PIECE, c * PIECE:(c + 1) * PIECE])
    # the dictionary of paths
    return paths

# ---------- prepare the files for one chunk ----------
# full size: cut every image into pieces; items = (id, image file, mask file, OCR file)
def make_full_items(rows):
    # the items and the layout of each image
    items, layout = [], {}
    # each image
    for r in rows:
        # read the decoded pixels (BGR, like the loader)
        img = cv2.imread(r["abs_path"], cv2.IMREAD_COLOR)
        # it must be the size the manifest says
        assert img.shape[:2] == (r["H"], r["W"]), "unexpected image size for " + r["path"]
        # pieces per side
        n_r, n_c = r["n_rows"], r["n_cols"]
        # pad with white to a multiple of 512
        padded = pad_to(img, n_r * PIECE, n_c * PIECE, 255)
        # the OCR pieces of this page
        op = ocr_pieces_for(r["page"], r["W"], r["H"])
        # ids of the pieces of this image
        tiles = []
        # rows
        for rr in range(n_r):
            # columns
            for cc in range(n_c):
                # a unique id without dots (infer.py names its maps by the file name before the first dot)
                tid = "f%05d_r%d_c%d" % (r["idx"], rr, cc)
                # the piece file
                pf = os.path.join(WORK_DIR, "piece_img", tid + ".png")
                # write the piece (lossless)
                cv2.imwrite(pf, padded[rr * PIECE:(rr + 1) * PIECE, cc * PIECE:(cc + 1) * PIECE])
                # one item
                items.append((tid, pf, ZERO_MASK, op[(rr, cc)]))
                # remember the id
                tiles.append(tid)
        # the layout used to stitch
        layout[r["idx"]] = tiles
    # done
    return items, layout

# shrink512: one link per image with a short unique name; the loader reads the real file through it
def make_shrink_items(rows):
    # the items and the layout
    items, layout = [], {}
    # each image
    for r in rows:
        # a unique id and a link name with the real extension
        tid = "s%05d" % r["idx"]
        # the link path
        link = os.path.join(WORK_DIR, "links", tid + os.path.splitext(r["abs_path"])[1])
        # replace an old link
        if os.path.lexists(link):
            # remove it
            os.remove(link)
        # make the link (absolute path of the real file)
        os.symlink(os.path.abspath(r["abs_path"]), link)
        # one item: the real box mask is used (the loader resizes it with the page)
        items.append((tid, link, r["abs_mask"], OCR_MASKS[r["page"]]))
        # the layout
        layout[r["idx"]] = [tid]
    # done
    return items, layout

# ---------- run infer.py on both GPUs ----------
# the lines of the last model-load report: "model ready ... | missing keys: N | unexpected keys: M"
LOAD_REPORTS = []

# run the items on all GPUs (or on `gpus` of them); returns {id: seconds}
def run_chunk(tag, items, gpus=None, limit=0):
    # how many GPUs to use
    gpus = gpus or N_GPUS
    # where this chunk's probability maps go
    out_dir = os.path.join(WORK_DIR, "probs", tag)
    # make it
    os.makedirs(out_dir, exist_ok=True)
    # the background jobs
    jobs = []
    # one job per GPU
    for k in range(gpus):
        # every gpus-th item, starting at k (cards are dealt like playing cards)
        part = items[k::gpus]
        # nothing for this GPU
        if not part:
            continue
        # the path-list name (also the csv and log name)
        ds_name = tag + "_gpu" + str(k)
        # save the path list that infer.py reads
        with open(os.path.join(PKL_DIR, ds_name + ".pkl"), "wb") as f:
            # (image, mask, OCR) triples
            pickle.dump([(i, m, o) for (_, i, m, o) in part], f)
        # old score table of the same name would be appended to: remove it
        csv_path = os.path.join(SCORES_DIR, ds_name + ".csv")
        # remove it
        if os.path.exists(csv_path):
            os.remove(csv_path)
        # the log file
        log_path = os.path.join(SCORES_DIR, ds_name + ".log")
        # open it
        lf = open(log_path, "w")
        # start the job in the background
        proc = subprocess.Popen(infer_cmd(ds_name, out_dir, csv_path, limit), stdout=lf, stderr=subprocess.STDOUT, env=gpu_env(k))
        # remember it
        jobs.append(dict(gpu=k, proc=proc, log=log_path, lf=lf, csv=csv_path))
    # wait for all jobs; print a progress line about once a minute
    t0, last = time.time(), time.time()
    # while any job runs
    while any(j["proc"].poll() is None for j in jobs):
        # short sleep
        time.sleep(2)
        # progress
        if time.time() - last > 60:
            # count the maps
            print("   ... %.1f min, %d maps so far in %s" % ((time.time() - t0) / 60, len(glob.glob(out_dir + "/*.npy")), tag), flush=True)
            # reset
            last = time.time()
    # close the logs and check every job
    seconds = {}
    # each job
    for j in jobs:
        # close the log
        j["lf"].close()
        # read the log
        text = open(j["log"]).read()
        # failed: show the last 30 lines and stop
        if j["proc"].returncode != 0:
            # show
            print("".join(text.splitlines(True)[-30:]))
            # stop
            raise RuntimeError("infer.py failed on GPU " + str(j["gpu"]) + " (last 30 lines above)")
        # the model-load report
        for m in re.finditer(r"missing keys: (\d+) \| unexpected keys: (\d+)", text):
            # remember
            LOAD_REPORTS.append((tag, j["gpu"], int(m.group(1)), int(m.group(2))))
        # the seconds of each image from the score table
        if os.path.exists(j["csv"]):
            # read it
            t = pd.read_csv(j["csv"])
            # id -> seconds
            for name, s in zip(t["image"], t["seconds"]):
                # store
                seconds[str(name)] = float(s)
    # give the seconds back
    return seconds

# ---------- put the maps back together ----------
# one image: load its maps and build the page-size 8-bit map
def stitch(size, r, tiles, probs_dir):
    # full size: place each 512 x 512 piece on a canvas and crop the padding
    if size == "full":
        # the canvas
        canvas = np.zeros((r["n_rows"] * PIECE, r["n_cols"] * PIECE), np.float32)
        # each piece
        for tid in tiles:
            # its position is in its id: ..._r<row>_c<col>
            rr, cc = [int(x[1:]) for x in tid.split("_")[1:3]]
            # read the probability map (float16 on disk)
            p = np.load(os.path.join(probs_dir, tid + ".npy")).astype(np.float32)
            # it must be one piece
            assert p.shape == (PIECE, PIECE), "piece map has shape " + str(p.shape)
            # place it
            canvas[rr * PIECE:(rr + 1) * PIECE, cc * PIECE:(cc + 1) * PIECE] = p
        # crop the padding
        return to8(canvas[:r["H"], :r["W"]])
    # shrink512: one map at the shrunk size
    p = np.load(os.path.join(probs_dir, tiles[0] + ".npy")).astype(np.float32)
    # not the page size: resize back with bilinear interpolation (width, height)
    if p.shape != (r["H"], r["W"]):
        # resize
        p = cv2.resize(p, (r["W"], r["H"]), interpolation=cv2.INTER_LINEAR)
    # 8-bit
    return to8(p)

# delete the temporary files of one chunk
def cleanup(tag, size):
    # probability maps
    shutil.rmtree(os.path.join(WORK_DIR, "probs", tag), ignore_errors=True)
    # pieces and links
    for d in ["piece_img", "links"]:
        # remove the files and make the folder again
        shutil.rmtree(os.path.join(WORK_DIR, d), ignore_errors=True)
        # re-create
        os.makedirs(os.path.join(WORK_DIR, d), exist_ok=True)

## 6. Setup check and a timed estimate (this is also the model-load test)

**Why (ELI5):** before the long run we run the model on a handful of real pieces and images, on one GPU, and time them. This
proves three things: the model loads with **0 missing keys** (all weights found), it runs on real data, and how long the
whole run will take. If the estimate is above `MAX_RUN_MINUTES` the notebook stops instead of starting a run we did not
plan for (a SMOKE run only prints the projection for the full run).

*Known-answer check:* the Week 5 notebook already reproduced the paper's number for this model on T-SROIE; here the
box-versus-outside table at the end is the sanity look on our own data.

**Estimate** = (pieces × seconds per piece + images × seconds per shrunk image) ÷ GPUs + one model load per chunk
+ preparing and stitching time.

In [ ]:
# images still to do at each size
todo_full = todo_rows("full") if "full" in SIZES else []
# shrink512
todo_shrink = todo_rows("shrink512") if "shrink512" in SIZES else []
# pieces still to run
n_pieces_total = int(sum(r["n_pieces"] for r in todo_full))
# show the amount of work
print("still to do: full %d images (%d pieces) | shrink512 %d images" % (len(todo_full), n_pieces_total, len(todo_shrink)))

# defaults if a size has nothing to do
per_piece = per_shrink = load_s = prep_per_piece = 0.0
# a timing sample of full-size pieces
if todo_full:
    # prepare the first 2 images (time it)
    t0 = time.time()
    # items
    items_f, lay_f = make_full_items(todo_full[:2])
    # prepare seconds per piece
    prep_per_piece = (time.time() - t0) / max(1, len(items_f))
    # run them on one GPU (also tests the model load)
    t0 = time.time()
    # run
    secs_f = run_chunk("est_full", items_f, gpus=1)
    # wall time of the run
    wall_f = time.time() - t0
    # seconds per piece (the first one includes GPU warm-up, so drop it when there are more)
    vals = list(secs_f.values())
    # median
    per_piece = float(np.median(vals[1:] if len(vals) > 1 else vals))
    # model load = wall time minus the time spent on pieces
    load_s = max(0.0, wall_f - sum(vals))
    # clean up
    cleanup("est_full", "full")
# a timing sample of shrunk images
if todo_shrink:
    # items
    items_s, lay_s = make_shrink_items(todo_shrink[:10])
    # run
    t0 = time.time()
    # run on one GPU
    secs_s = run_chunk("est_shrink", items_s, gpus=1)
    # wall
    wall_s = time.time() - t0
    # values
    vals = list(secs_s.values())
    # seconds per image
    per_shrink = float(np.median(vals[1:] if len(vals) > 1 else vals))
    # model load
    load_s = max(load_s, wall_s - sum(vals))
    # clean up
    cleanup("est_shrink", "shrink512")

# the model must have loaded with 0 missing keys on every GPU job so far
bad_load = [x for x in LOAD_REPORTS if x[2] != 0]
# hard check
check("model loads with 0 missing keys", str(LOAD_REPORTS[:4]) if LOAD_REPORTS else "no load report found", bool(LOAD_REPORTS) and not bad_load)
# stop if not
stop_if_failed()

# chunks (each chunk = one model load per GPU)
n_chunks = ceil_div(len(todo_full), CHUNK_IMAGES_FULL) + ceil_div(len(todo_shrink), CHUNK_IMAGES_SHRINK)
# minutes of GPU work
gpu_min = (n_pieces_total * per_piece + len(todo_shrink) * per_shrink) / N_GPUS / 60
# minutes of model loading
load_min = n_chunks * load_s / 60
# minutes of preparing pieces and stitching (done on the CPU, not in parallel with the GPUs)
prep_min = prep_per_piece * 1.5 * n_pieces_total / 60 + len(todo_shrink) * 0.05 / 60
# the total
est_min = gpu_min + load_min + prep_min
# show it
print("seconds per piece %.3f | per shrunk image %.3f | model load %.0f s | GPUs %d" % (per_piece, per_shrink, load_s, N_GPUS))
print("ESTIMATE for what is left: %.1f min (GPU %.1f + model loads %.1f + preparing/stitching %.1f), %d chunks" % (est_min, gpu_min, load_min, prep_min, n_chunks))
# a smoke run: project to the full run (all images of the real dataset)
if SMOKE and len(images):
    # scale by the number of images
    scale = EXPECTED_IMAGES_FULL_RUN / len(images)
    # chunks in a full run
    n_chunks_full_run = ceil_div(EXPECTED_IMAGES_FULL_RUN, CHUNK_IMAGES_FULL) + ceil_div(EXPECTED_IMAGES_FULL_RUN, CHUNK_IMAGES_SHRINK)
    # minutes
    proj = (gpu_min + prep_min) * scale + n_chunks_full_run * load_s / 60
    # show it
    print("PROJECTION for the full run (about %d images, same mix of page sizes as these %d): about %.0f min (+ about 8 min environment, + OCR). Approved up to %d min."
          % (EXPECTED_IMAGES_FULL_RUN, len(images), proj, MAX_RUN_MINUTES))
    # remember
    log_time("estimate: full-run projection (seconds)", proj * 60, "from the SMOKE sample: %.0f min" % proj)
# remember the estimate
log_time("estimate for this run (seconds)", est_min * 60, "%.3f s/piece, %.3f s/shrunk image" % (per_piece, per_shrink))
# too long: stop (a smoke run never stops here)
if not SMOKE and est_min > MAX_RUN_MINUTES:
    # message
    raise RuntimeError("STOP: estimate %.0f min is over MAX_RUN_MINUTES = %d. Raise it in section 1 only if you accept the time." % (est_min, MAX_RUN_MINUTES))

## 7. The run (resumable)

**Why (ELI5):** now every image goes through ADCD-Net. For each chunk of images we prepare the files, run both GPUs, stitch
the maps back, save the final 8-bit maps in the standard folders, and delete the temporary files. If the notebook stops, run
it again: images that already have a map are skipped. Progress is printed after every chunk.

In [ ]:
# one size: chunk after chunk
def process_size(size):
    # the images that still have no map
    rows = todo_rows(size)
    # nothing to do
    if not rows:
        # say so
        print(size, ": nothing to do (all maps exist)")
        return
    # images per chunk
    n = CHUNK_IMAGES_FULL if size == "full" else CHUNK_IMAGES_SHRINK
    # the chunks
    chunks = [rows[i:i + n] for i in range(0, len(rows), n)]
    # stopwatch of this size
    t_size = time.time()
    # each chunk
    for ci, chunk in enumerate(chunks, 1):
        # stopwatch of this chunk
        t0 = time.time()
        # a unique name for the chunk
        tag = "%s_%03d" % (size, ci)
        # prepare the files
        items, layout = make_full_items(chunk) if size == "full" else make_shrink_items(chunk)
        # run both GPUs
        secs = run_chunk(tag, items)
        # the folder with this chunk's maps
        probs_dir = os.path.join(WORK_DIR, "probs", tag)
        # stitch and save every image
        for r in chunk:
            # its piece ids
            tiles = layout[r["idx"]]
            # the page-size 8-bit map
            m8 = stitch(size, r, tiles, probs_dir)
            # seconds of this image = sum of its pieces
            s = sum(secs.get(t, 0.0) for t in tiles)
            # save it and log it
            save_result(size, r, m8, s)
        # delete the temporary files
        cleanup(tag, size)
        # progress
        done = sum(len(c) for c in chunks[:ci])
        # print
        print("%s chunk %d/%d: %d images, %d pieces, %.1f min | %d/%d images done, %.1f min so far" %
              (size, ci, len(chunks), len(chunk), len(items), (time.time() - t0) / 60, done, len(rows), (time.time() - t_size) / 60), flush=True)
    # record the time of this size
    log_time("run " + size, time.time() - t_size, str(len(rows)) + " images")

# the whole run
t_run = time.time()
# each size
for size in SIZES:
    # run it
    process_size(size)
# record the time
log_time("run total", time.time() - t_run, str(len(images)) + " images x " + str(len(SIZES)) + " sizes")
# the model-load reports of the whole run must still be clean
check("model loaded with 0 missing keys in every job", str(len(LOAD_REPORTS)) + " reports, " + str(len([x for x in LOAD_REPORTS if x[2] != 0])) + " with missing keys", all(x[2] == 0 for x in LOAD_REPORTS))

## 8. Checks, `run_info.json`, timing and the three zips

**Why (ELI5):** a last look that every image has a map at every size, that every map has the page's exact size and is 8-bit,
and that the run log has one row per image and size. Any gap **stops** the notebook. Then three zips:
**(1) all outputs**; **(2) safe for Claude** (tables only: no maps, no masks, no images, no statement text); **(3) next
round** (the folder `adcd/` that notebook 3 reads: upload it as the dataset `adcd-results`).

The table `sanity_box_vs_outside` is a quick look at whether the maps make sense: edited versions should have a clearly higher
mean value inside the box than outside it.

In [ ]:
# ---------- page-count checks: every image, every size ----------
# one row per size: expected, found, missing, wrong size
count_rows = []
# problems found
bad_size, bad_mode = [], []
# each size
for size in SIZES:
    # missing results
    missing = 0
    # each image
    for r in images.to_dict("records"):
        # the file
        p = result_path(size, r)
        # missing
        if not os.path.exists(p):
            # count
            missing += 1
            continue
        # read the header only
        im = Image.open(p)
        # same size as the page?
        if im.size != (r["W"], r["H"]):
            # remember
            bad_size.append(size + ":" + r["path"])
        # 8-bit greyscale?
        if im.mode != "L":
            # remember
            bad_mode.append(size + ":" + r["path"])
    # the row
    count_rows.append(dict(size=size, expected=len(images), found=len(images) - missing, missing=missing))
# show the counts
print(pd.DataFrame(count_rows).to_string(index=False))
# result files per size, set, version and format
res_counts = []
# each size
for size in SIZES:
    # count the files by their position in the standard layout
    for r in images.to_dict("records"):
        # only files that exist
        if os.path.exists(result_path(size, r)):
            # one row
            res_counts.append(dict(size=size, img_set=r["img_set"], version=r["version"], format=r["format"]))
# the table
print(pd.DataFrame(res_counts).groupby(["size", "img_set", "version", "format"]).size().unstack(["size", "format"], fill_value=0).to_string())
# the hard checks
check("every image has a result at every size", str([c for c in count_rows if c["missing"] > 0]), all(c["missing"] == 0 for c in count_rows))
# size check
check("every result map has the page size", str(len(bad_size)) + " bad " + str(bad_size[:3]), len(bad_size) == 0)
# mode check
check("every result map is 8-bit greyscale", str(len(bad_mode)) + " bad " + str(bad_mode[:3]), len(bad_mode) == 0)

# ---------- the run log: one row per image and size ----------
# read it
log = pd.read_csv(RUN_LOG)
# images that have a result file but no log row (a crash between the two writes): add them
have = set(zip(log["image"], log["size"]))
# the missing rows
extra = []
# each size and image
for size in SIZES:
    # each image
    for r in images.to_dict("records"):
        # a log row exists
        if (r["path"], size) in have:
            continue
        # read the map to get its max value
        m = cv2.imread(result_path(size, r), cv2.IMREAD_GRAYSCALE)
        # a row
        extra.append(dict(image=r["path"], size=size, set=r["set"], version=r["version"], format=r["format"], page=r["page"], seconds=np.nan, max_value=int(m.max()), status="existing"))
# add them
if extra:
    # extend the log and save it
    log = pd.concat([log, pd.DataFrame(extra)], ignore_index=True)
    # write
    log.to_csv(RUN_LOG, index=False)
# keep only the last row of every (image, size)
log = log.drop_duplicates(["image", "size"], keep="last")
# save the clean log
log.to_csv(RUN_LOG, index=False)
# one row per image and size is required
check("run_log.csv has one row per image and size", str(len(log)) + " rows, expected " + str(len(images) * len(SIZES)), len(log) == len(images) * len(SIZES))
# all rows ok
check("no failed images in the run log", str(int((log["status"] != "ok").sum() - (log["status"] == "existing").sum())) + " not ok", int(((log["status"] != "ok") & (log["status"] != "existing")).sum()) == 0)
# seconds per image and size
print(log.groupby("size")["seconds"].agg(["count", "mean", "median", "max"]).round(3).to_string())

# ---------- a sanity look: mean 8-bit value inside the box versus outside (information, not a check) ----------
# a seeded sample of at most 150 edited images
rng = np.random.default_rng(SEED)
# the edited ones
edited = images[images["version"] != "untouched"]
# sample
sample = edited.iloc[rng.permutation(len(edited))[:150]]
# rows
san = []
# each sampled image and size
for r in sample.to_dict("records"):
    # box mask
    bm = cv2.imread(r["abs_mask"], cv2.IMREAD_GRAYSCALE) > 127
    # each size
    for size in SIZES:
        # the result map
        m = cv2.imread(result_path(size, r), cv2.IMREAD_GRAYSCALE)
        # one row
        san.append(dict(size=size, version=r["version"], format=r["format"], in_box=float(m[bm].mean()), outside=float(m[~bm].mean()), flagged_in_box=bool((m[bm] >= 128).any())))
# the summary table
sanity = pd.DataFrame(san).groupby(["size", "version", "format"]).agg(n=("in_box", "count"), mean_in_box=("in_box", "mean"), mean_outside=("outside", "mean"), flagged_in_box=("flagged_in_box", "mean")).round(3).reset_index()
# show it
print(sanity.to_string(index=False))
# save it
sanity.to_csv(os.path.join(RUN_DIR, "sanity_box_vs_outside.csv"), index=False)

# ---------- run_info.json (sits directly in results/<detector>/) ----------
# versions line from the environment check, if there is one
env_txt = open(os.path.join(RUN_DIR, "env_versions.txt")).read().strip() if os.path.exists(os.path.join(RUN_DIR, "env_versions.txt")) else ""
# the information
run_info = {"detector": DETECTOR, "weights_files": [os.path.basename(CKPT_PATH), os.path.basename(DOCRES_PATH)], "piece_size": PIECE, "sizes": SIZES, "n_images": int(len(images)), "n_results": int(len(log)),
            "threshold_for_flagging": 0.5, "value_scale": "round(255 x probability of tampered)", "seed": SEED, "SMOKE": SMOKE,
            "python_and_package_versions": env_txt.splitlines()[:6], "notes": ["official ADCD-Net loader: long side shrunk to 512 (shrink512); every input converted to greyscale and re-saved as JPEG quality 100 inside the loader to get DCT numbers (JPEG history of the input is not used); full = 512 x 512 pieces on the page grid from (0,0), padded white right/bottom, pieces saved as PNG; OCR masks: Week 5 for TextCtrl, PaddleOCR on the untouched page for the 49 shared pages, one mask per page for all versions; shrink512 maps resized back with bilinear interpolation"], "date": datetime.now().isoformat(timespec="seconds")}
# write it
with open(os.path.join(RES_DIR, "run_info.json"), "w") as f:
    # save
    json.dump(run_info, f, indent=2)
# show it
print(json.dumps(run_info, indent=1)[:1500])

# ---------- checks table and timing table ----------
# save the checks
pd.DataFrame(CHECKS).to_csv(os.path.join(RUN_DIR, "checks.csv"), index=False)
# save the timing
pd.DataFrame(TIMING).to_csv(os.path.join(RUN_DIR, "timing.csv"), index=False)
# show both
print(pd.DataFrame(TIMING).to_string(index=False))
# stop if any hard check failed
stop_if_failed()

# ---------- the three zips ----------
# make a zip from (file, name inside zip) pairs
def make_zip(zip_path, items):
    # write the zip
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        # each file
        for src, arc in items:
            # add it
            z.write(src, arc)
    # size in MB
    return os.path.getsize(zip_path) / 1e6

# every file under a folder as (path, name in zip)
def walk_items(folder, prefix):
    # the list
    items = []
    # walk
    for root, dirs, fs in os.walk(folder):
        # each file
        for f in fs:
            # skip leftovers
            if f.endswith("_tmp.png") or f == ".DS_Store":
                continue
            # full path
            full = os.path.join(root, f)
            # name inside
            items.append((full, os.path.join(prefix, os.path.relpath(full, folder))))
    # done
    return items

# (3) NEXT ROUND: the result folder results/<detector>/ (what notebook 3 reads), top folder = the detector name
next_items = walk_items(RES_DIR, DETECTOR)
# (1) ALL outputs: the result folder plus every log, config, table and figure of this notebook (and the OCR masks if there are any)
all_items = walk_items(RES_DIR, "results/" + DETECTOR) + walk_items(RUN_DIR, "w7_" + NB_TAG + "_run")
# extra folders that exist
for extra_dir, name in EXTRA_ALL_DIRS:
    # only if it exists
    if os.path.isdir(extra_dir):
        # add its files
        all_items += walk_items(extra_dir, name)
# (2) SAFE for Claude: tables and text only (no maps, no masks, no images); the run log has page ids but no text from the statements
safe_files = [(RUN_LOG, "w7_safe/run_log.csv"), (os.path.join(RES_DIR, "run_info.json"), "w7_safe/run_info.json")]
# the tables and logs of the run folder
for f in ["checks.csv", "timing.csv", "config.json", "env_versions.txt", "sanity_box_vs_outside.csv"] + EXTRA_SAFE_FILES:
    # only if it exists
    if os.path.exists(os.path.join(RUN_DIR, f)):
        # add
        safe_files.append((os.path.join(RUN_DIR, f), "w7_safe/" + f))
# the zip paths (names tell what is inside)
ZIPS = {"1_ALL_outputs": os.path.join(WORKING, "w7_" + NB_TAG + "_1_ALL_outputs.zip"),
        "2_SAFE_for_claude": os.path.join(WORKING, "w7_" + NB_TAG + "_2_SAFE_for_claude.zip"),
        "3_NEXT_ROUND": os.path.join(WORKING, "w7_" + NB_TAG + "_3_NEXT_ROUND_" + DETECTOR + "-results.zip")}
# build them
for key, items in [("1_ALL_outputs", all_items), ("2_SAFE_for_claude", safe_files), ("3_NEXT_ROUND", next_items)]:
    # make and report
    mb = make_zip(ZIPS[key], items)
    # print
    print("%-20s %8.1f MB %6d files  %s" % (key, mb, len(items), ZIPS[key]))
# download links (relative to the working folder on Kaggle)
try:
    # IPython
    from IPython.display import FileLink, display
    # relative links need the working folder as the current folder
    os.chdir(WORKING)
    # each zip
    for key, path in ZIPS.items():
        # label
        print(key)
        # link
        display(FileLink(os.path.relpath(path, WORKING)))
# not in a notebook
except Exception as e:
    # print the paths
    print("no download links here:", e)
# the reminder
print("Upload zip 3 as a Kaggle dataset named " + DETECTOR + "-results (it unzips to a folder called " + DETECTOR + "/ with run_info.json inside).")
# smoke reminder
if SMOKE:
    # banner
    print("#" * 78)
    print("#  SMOKE=True: this was only a test. Set SMOKE=False before Save & Run All.")
    print("#" * 78)
print("DONE.", DETECTOR, "results:", RES_DIR)